# RAG Example: "What is the best performance/cost item sold last week?"

A realistic data flow showing how RAG answers a business question.

In [ ]:
import numpy as np
from datetime import datetime, timedelta
np.random.seed(42)

# ═══════════════════════════════════════════════════════════════
# STEP 0: The Company's Data (exists in their systems already)
# ═══════════════════════════════════════════════════════════════

# Product catalog (from SQL database)
products = [
    {"id": "P001", "name": "ThinkPad X1 Carbon",    "category": "laptop",    "price": 1400, "benchmark_score": 8500},
    {"id": "P002", "name": "MacBook Air M3",        "category": "laptop",    "price": 1100, "benchmark_score": 9200},
    {"id": "P003", "name": "Dell XPS 15",           "category": "laptop",    "price": 1800, "benchmark_score": 9000},
    {"id": "P004", "name": "HP Pavilion 15",        "category": "laptop",    "price": 600,  "benchmark_score": 5500},
    {"id": "P005", "name": "ASUS ROG Strix G16",    "category": "laptop",    "price": 1600, "benchmark_score": 11000},
    {"id": "P006", "name": "Acer Swift Go 14",      "category": "laptop",    "price": 750,  "benchmark_score": 7200},
    {"id": "P007", "name": "Samsung Galaxy S24",    "category": "phone",     "price": 800,  "benchmark_score": 6800},
    {"id": "P008", "name": "iPhone 15 Pro",         "category": "phone",     "price": 1000, "benchmark_score": 7500},
    {"id": "P009", "name": "Pixel 8",               "category": "phone",     "price": 500,  "benchmark_score": 5900},
    {"id": "P010", "name": "iPad Air",              "category": "tablet",    "price": 600,  "benchmark_score": 7800},
]

# Sales data (from last week)
today = datetime(2026, 5, 5)
last_week_start = today - timedelta(days=7)

sales = [
    {"product_id": "P001", "date": "2026-04-29", "qty": 12},
    {"product_id": "P002", "date": "2026-04-30", "qty": 45},
    {"product_id": "P003", "date": "2026-04-28", "qty": 8},
    {"product_id": "P004", "date": "2026-05-01", "qty": 30},
    {"product_id": "P005", "date": "2026-05-02", "qty": 15},
    {"product_id": "P006", "date": "2026-04-29", "qty": 22},
    {"product_id": "P007", "date": "2026-05-01", "qty": 35},
    {"product_id": "P008", "date": "2026-05-03", "qty": 28},
    {"product_id": "P009", "date": "2026-05-02", "qty": 40},
    {"product_id": "P010", "date": "2026-04-30", "qty": 18},
]

print("Company Data (exists in their SQL database):\n")
print(f"  Products: {len(products)} items")
print(f"  Sales last week: {sum(s['qty'] for s in sales)} units sold")
print(f"  Date range: {last_week_start.strftime('%Y-%m-%d')} to {today.strftime('%Y-%m-%d')}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 1: INDEXING (done once, or refreshed nightly)
#         Application code queries SQL, computes metrics,
#         creates text chunks, embeds them, stores in vector DB
# ═══════════════════════════════════════════════════════════════

print("STEP 1: Indexing (runs nightly via cron job)\n")
print("─" * 60)

# 1a. Application code queries the SQL database and computes derived metrics
print("\n  1a. Query SQL, compute metrics:\n")

product_summaries = []
for product in products:
    # Find sales for this product
    product_sales = [s for s in sales if s['product_id'] == product['id']]
    total_qty = sum(s['qty'] for s in product_sales)
    
    # Compute performance/cost ratio
    perf_cost_ratio = product['benchmark_score'] / product['price']
    revenue = total_qty * product['price']
    
    product_summaries.append({
        **product,
        "qty_sold_last_week": total_qty,
        "revenue_last_week": revenue,
        "perf_cost_ratio": round(perf_cost_ratio, 2),
    })

# Sort by perf/cost ratio
product_summaries.sort(key=lambda x: -x['perf_cost_ratio'])

print(f"  {'Product':<22} {'Price':>6} {'Bench':>6} {'Perf/$':>7} {'Sold':>5} {'Revenue':>9}")
print(f"  {'─'*22} {'─'*6} {'─'*6} {'─'*7} {'─'*5} {'─'*9}")
for p in product_summaries:
    print(f"  {p['name']:<22} ${p['price']:>5} {p['benchmark_score']:>6} {p['perf_cost_ratio']:>7.2f} {p['qty_sold_last_week']:>5} ${p['revenue_last_week']:>8,}")

In [ ]:
# 1b. Convert structured data into TEXT CHUNKS (this is the key step!)
#     The LLM can only read text, so we must describe the data in natural language.

print("  1b. Convert to text chunks (LLM needs text, not SQL rows):\n")

chunks = []

# Chunk 1: Overall sales summary
total_revenue = sum(p['revenue_last_week'] for p in product_summaries)
total_units = sum(p['qty_sold_last_week'] for p in product_summaries)
chunk_summary = (
    f"Weekly Sales Summary (2026-04-28 to 2026-05-04): "
    f"Total revenue: ${total_revenue:,}. Total units sold: {total_units}. "
    f"Top seller by volume: {max(product_summaries, key=lambda x: x['qty_sold_last_week'])['name']} "
    f"({max(product_summaries, key=lambda x: x['qty_sold_last_week'])['qty_sold_last_week']} units). "
    f"Top revenue item: {max(product_summaries, key=lambda x: x['revenue_last_week'])['name']} "
    f"(${max(product_summaries, key=lambda x: x['revenue_last_week'])['revenue_last_week']:,})."
)
chunks.append(chunk_summary)

# Chunk 2: Performance/cost analysis
best_perf_cost = product_summaries[0]  # already sorted
chunk_perf = (
    f"Performance/Cost Analysis (week of 2026-04-28): "
    f"Best performance per dollar among items sold last week: "
    f"{best_perf_cost['name']} (benchmark: {best_perf_cost['benchmark_score']}, "
    f"price: ${best_perf_cost['price']}, ratio: {best_perf_cost['perf_cost_ratio']} points/$). "
    f"Second best: {product_summaries[1]['name']} (ratio: {product_summaries[1]['perf_cost_ratio']}). "
    f"Third: {product_summaries[2]['name']} (ratio: {product_summaries[2]['perf_cost_ratio']})."
)
chunks.append(chunk_perf)

# Chunk 3-N: Individual product details
for p in product_summaries:
    chunk = (
        f"Product: {p['name']}. Category: {p['category']}. Price: ${p['price']}. "
        f"Benchmark score: {p['benchmark_score']}. Performance/cost ratio: {p['perf_cost_ratio']}. "
        f"Units sold last week: {p['qty_sold_last_week']}. Revenue last week: ${p['revenue_last_week']:,}."
    )
    chunks.append(chunk)

print(f"  Created {len(chunks)} text chunks:\n")
for i, chunk in enumerate(chunks[:4]):
    print(f"  Chunk {i}: '{chunk[:80]}...'")
print(f"  ... ({len(chunks) - 4} more product chunks)")

In [ ]:
# 1c. Embed chunks and store in vector DB

print("  1c. Embed chunks → store in vector DB:\n")

# In real life: embedding_api.embed(chunk) → 768-dim vector
# Here we simulate with random vectors (in reality, semantically similar text → similar vectors)
embedding_dim = 32

# Simulate: chunks about "performance/cost" get similar embeddings
def fake_embed(text):
    np.random.seed(hash(text) % 2**31)
    vec = np.random.randn(embedding_dim)
    # Boost dimensions if text contains key terms (simulates semantic meaning)
    if 'performance' in text.lower() or 'ratio' in text.lower():
        vec[0:4] += 2.0  # "performance" cluster
    if 'sold' in text.lower() or 'revenue' in text.lower():
        vec[4:8] += 1.5  # "sales" cluster
    if 'last week' in text.lower() or '2026-04' in text:
        vec[8:12] += 1.5  # "recent" cluster
    return vec / np.linalg.norm(vec)

# Vector DB (simple list for demo)
vector_db = [(chunk, fake_embed(chunk)) for chunk in chunks]

print(f"  Vector DB now contains {len(vector_db)} entries")
print(f"  Each entry: (text_chunk, {embedding_dim}-dim vector)")
print(f"\n  In production:")
print(f"    embedding = openai.embed(chunk)  # or sentence-transformers")
print(f"    pinecone.upsert(id=chunk_id, vector=embedding, metadata=chunk_text)")
print(f"\n  This indexing runs nightly. Now the data is searchable.")
print(f"\n{'═' * 60}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 2: USER ASKS A QUESTION (happens in real-time)
# ═══════════════════════════════════════════════════════════════

user_query = "What is the best performance/cost item sold last week?"

print("STEP 2: User Query (real-time)\n")
print("─" * 60)
print(f"\n  User: '{user_query}'")
print(f"\n  The LLM does NOT have access to our sales database.")
print(f"  It was trained months ago. It doesn't know our products.")
print(f"  We need to FIND the relevant data and GIVE it to the LLM.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 3: RETRIEVAL (application code searches vector DB)
# ═══════════════════════════════════════════════════════════════

print("STEP 3: Retrieval (find relevant chunks)\n")
print("─" * 60)

# 3a. Embed the user's question
query_vector = fake_embed(user_query)
print(f"\n  3a. Embed query: '{user_query}'")
print(f"      → vector of {embedding_dim} dimensions")

# 3b. Search vector DB (cosine similarity)
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

scores = [(chunk, cosine_sim(query_vector, vec)) for chunk, vec in vector_db]
scores.sort(key=lambda x: -x[1])

# Take top 3
top_k = 3
retrieved = scores[:top_k]

print(f"\n  3b. Search vector DB (top {top_k} by cosine similarity):\n")
for i, (chunk, score) in enumerate(retrieved):
    print(f"  Result {i+1} [sim={score:.3f}]:")
    print(f"    '{chunk[:100]}...'\n")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 4: GENERATION (build prompt, call LLM)
# ═══════════════════════════════════════════════════════════════

print("STEP 4: Build Prompt & Call LLM\n")
print("─" * 60)

# Build the final prompt (this is what the LLM actually sees)
context = "\n".join(f"[Doc {i+1}]: {chunk}" for i, (chunk, _) in enumerate(retrieved))

final_prompt = f"""You are a sales analytics assistant. Answer based ONLY on the provided context.
If the data doesn't contain the answer, say so.

Context:
{context}

Question: {user_query}

Answer:"""

print(f"  THE ACTUAL PROMPT SENT TO LLM:")
print(f"  {'┌' + '─' * 70 + '┐'}")
for line in final_prompt.split('\n'):
    print(f"  │ {line:<69}│")
print(f"  {'└' + '─' * 70 + '┘'}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 5: LLM RESPONSE (simulated — in reality, API call)
# ═══════════════════════════════════════════════════════════════

print("STEP 5: LLM Response\n")
print("─" * 60)

# This is what the LLM would generate (simulated)
llm_response = (
    f"Based on last week's sales data, the best performance/cost item is the "
    f"**{best_perf_cost['name']}** with a performance-to-cost ratio of "
    f"{best_perf_cost['perf_cost_ratio']} points per dollar "
    f"(benchmark: {best_perf_cost['benchmark_score']}, price: ${best_perf_cost['price']}).\n\n"
    f"It sold {best_perf_cost['qty_sold_last_week']} units last week, "
    f"generating ${best_perf_cost['revenue_last_week']:,} in revenue.\n\n"
    f"Runner-up: {product_summaries[1]['name']} (ratio: {product_summaries[1]['perf_cost_ratio']})."
)

print(f"\n  User: '{user_query}'\n")
print(f"  Assistant: {llm_response}")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# COMPLETE DATA FLOW SUMMARY
# ═══════════════════════════════════════════════════════════════

print("\n" + "═" * 60)
print("COMPLETE DATA FLOW SUMMARY")
print("═" * 60)

print("""
  NIGHTLY (indexing):
  ┌──────────┐    ┌──────────────┐    ┌──────────┐    ┌──────────┐
  │ SQL DB   │ →  │ App Code     │ →  │ Embed    │ →  │ Vector   │
  │ (sales,  │    │ (query,      │    │ API      │    │ DB       │
  │ products)│    │ compute,     │    │          │    │ (chunks  │
  │          │    │ make text)   │    │          │    │ +vectors)│
  └──────────┘    └──────────────┘    └──────────┘    └──────────┘

  REAL-TIME (per user query):
  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
  │ User     │ →  │ Embed    │ →  │ Vector   │ →  │ Build    │
  │ Question │    │ Question │    │ DB       │    │ Prompt   │
  │          │    │          │    │ (search) │    │          │
  └──────────┘    └──────────┘    └──────────┘    └────┬─────┘
                                                       ↓
                                                  ┌──────────┐
                                                  │ LLM      │
                                                  │ (reads   │
                                                  │ context, │
                                                  │ answers) │
                                                  └──────────┘

  WHO DOES WHAT:
    • SQL DB:      stores raw data (existing system)
    • App Code:    YOUR code — queries SQL, computes metrics, builds text
    • Embed API:   converts text → vectors (OpenAI, Cohere, etc.)
    • Vector DB:   stores and searches vectors (Pinecone, ChromaDB)
    • LLM:         reads the retrieved text, generates natural language answer

  THE LLM NEVER TOUCHES YOUR DATABASE.
  Your application code is the glue between all components.
""")